# Chapter 18: Advanced Time Series Topics

Topics: infinite distributed lags, unit root testing, spurious regression, cointegration, error correction, forecasting.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.arima.model import ARIMA
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 18.3 Spurious Regression

In [ ]:
np.random.seed(42)
T = 100
rw1 = np.cumsum(np.random.normal(0, 1, T))
rw2 = np.cumsum(np.random.normal(0, 1, T))

df_spurious = pd.DataFrame({'y': rw1, 'x': rw2})
m_spurious  = smf.ols('y ~ x', data=df_spurious).fit()

print(f"R² = {m_spurious.rsquared:.4f} (suspiciously high for unrelated random walks!)")
print(f"t-stat on x = {m_spurious.tvalues['x']:.4f}")
print(f"\nThis is spurious regression: both variables are I(1) random walks with no true relationship")

## 18.4 Cointegration Test

In [ ]:
df = woo.dataWoo('inven')
y1 = df['inven'].values
y2 = df['gdp'].values

score, pval, crit = coint(y1, y2)
print(f"Engle-Granger cointegration test:")
print(f"  stat={score:.4f}, p-value={pval:.4f}")
print(f"  Critical values: {crit}")
print("→ Cointegrated" if pval < 0.05 else "→ Not cointegrated")

## 18.5 Forecasting with ARIMA

In [ ]:
df = woo.dataWoo('phillips')
df.index = pd.date_range(start='1948', periods=len(df), freq='YE')

train = df['inf'][:-4]
test  = df['inf'][-4:]

arima = ARIMA(train, order=(1, 0, 1)).fit()
forecast = arima.forecast(steps=4)

plt.figure(figsize=(12, 5))
plt.plot(df['inf'], color='steelblue', label='Actual')
plt.plot(forecast.index, forecast, 'r--o', label='Forecast', lw=2)
plt.axvline(train.index[-1], color='gray', linestyle=':')
plt.title('ARIMA(1,0,1) Forecast: US Inflation')
plt.legend()
plt.savefig(f"{output_dir}/fig1_arima_forecast.png", dpi=150)
plt.show()

from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(test, forecast))
print(f"RMSE: {rmse:.4f}")

## VAR Model

In [ ]:
macro = df[['inf', 'unem']].dropna()
var_model = VAR(macro).fit(maxlags=4, ic='aic')
print(var_model.summary())

# Impulse Response Functions
irf = var_model.irf(10)
irf.plot(orth=False)
plt.savefig(f"{output_dir}/fig2_irf.png", dpi=150, bbox_inches='tight')
plt.show()